# When ASR confidence does not travel: reproduce the results

This notebook reproduces the study in [https://github.com/neric-joel/DL-Project-](https://github.com/neric-joel/DL-Project-) on Google Colab.

* **Part A (about 2 minutes, CPU is fine):** rebuild every table and figure from the per-unit results
  committed in the repository.
* **Part B (about 1 hour on a T4 GPU):** run the whole pipeline from the raw audio: download both
  datasets, run Whisper-small on 11,195 units, score, evaluate, and draw the figures.

For Part B choose *Runtime → Change runtime type → T4 GPU* first.

In [ ]:
# safe to re-run: clones once, then just updates
%cd /content
![ -d asrshift ] || git clone -q https://github.com/neric-joel/DL-Project- asrshift
%cd /content/asrshift
!git pull -q
!pip install -q -e .
# make the package importable in this kernel without a restart (re-run this cell after a restart)
import sys; sys.path.insert(0, "/content/asrshift/src")

## Part A: tables and figures from the committed per-unit results

Uses 500 bootstrap replicates to stay quick. Point estimates are identical to `results/`, and the
intervals and p-values differ slightly from the reported 2,000-replicate ones.

In [ ]:
!asrshift evaluate --n-boot 500
!asrshift figures
!asrshift report

In [ ]:
import pandas as pd
from IPython.display import Image, display

t = "results/tables/small"
asr = pd.read_csv(f"{t}/asr_summary.csv").set_index("set")
display(asr.loc[["eka_test", "pm_turn_test", "pm_window_test"],
                ["units", "hours", "wer", "err_rate", "number_acc", "negation_acc", "mean_conf"]].round(3))
cal = pd.read_csv(f"{t}/err/calibration.csv")
display(cal[cal.policy.isin(["P3", "P4"])][["eval", "policy", "base_rate", "mean_p", "ece", "brier", "citl"]].round(3))
ops = pd.read_csv(f"{t}/err/operating_points.csv")
display(ops[(ops.point == "budget_0.10") & ops.policy.isin(["P1", "P3", "P1-retuned", "P4"])]
        [["eval", "policy", "review_rate", "err_caught", "accepted_err_rate"]].round(3))

In [ ]:
for f in ["fig1_confidence_vs_error", "fig2_reliability", "fig3_risk_coverage", "fig4_threshold_transfer",
          "fig5_recalibration_data"]:
    display(Image(f"results/figures/{f}.png", width=900))

## Part B: the full pipeline from raw audio (T4 GPU)

Downloads about 2.1 GB (Eka parquet shards from the Hugging Face Hub, PriMock57 audio from GitHub
LFS), builds the speaker-disjoint and consultation-level splits, runs faster-whisper, and rewrites
everything under `results/`. Each step is resumable if the runtime disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!asrshift download
!asrshift prepare

In [ ]:
# about 1 hour on a T4, resumable. Part A's committed results are overwritten by the steps below.
!asrshift infer

In [ ]:
!asrshift score
!asrshift evaluate
!asrshift figures
!asrshift report

Compare this run with the committed results. Float16 decoding is not bit-identical across GPU
models. In our test on a T4, 445 of the 11,195 transcripts (4%, mostly 30-second windows) came out
differently from the committed RTX 3050 run and 42 labels flipped; the estimates moved by up to 0.03
and every hypothesis decision stayed the same.

In [ ]:
import io, subprocess
import pandas as pd

def committed(path):
    out = subprocess.run(["git", "show", f"HEAD:{path}"], capture_output=True, text=True, check=True).stdout
    return pd.read_csv(io.StringIO(out))

t = "results/tables/small"
cols = ["units", "wer", "err_rate", "mean_conf"]
display(pd.concat({"committed": committed(f"{t}/asr_summary.csv").set_index("set")[cols],
                   "this run": pd.read_csv(f"{t}/asr_summary.csv").set_index("set")[cols]}, axis=1).round(3))
cols = ["estimate", "ci_lo", "ci_hi", "p_holm", "supported"]
display(pd.concat({"committed": committed(f"{t}/err/hypotheses.csv").set_index("hypothesis")[cols],
                   "this run": pd.read_csv(f"{t}/err/hypotheses.csv").set_index("hypothesis")[cols]}, axis=1).round(3))

## Part C (optional): a larger model

The proposal's stretch goal: does whisper-large-v3-turbo's confidence travel better? The repository
already contains this run (`results/SUMMARY_large-v3-turbo.md`, declared in protocol amendment 5). To
reproduce it from audio, allow about 2 to 3 hours on a T4. Each step is resumable.

In [ ]:
# pinned model revision and identical protocol, see configs/large-v3-turbo.yaml
!asrshift --config configs/large-v3-turbo.yaml infer
!asrshift --config configs/large-v3-turbo.yaml score
!asrshift --config configs/large-v3-turbo.yaml evaluate
!asrshift --config configs/large-v3-turbo.yaml report

## One transcript, end to end

In [ ]:
import pandas as pd
from asrshift import audio, infer, prepare, review
from asrshift.config import load_config

cfg = load_config()
m = prepare.load()
row = m[(m.subset == "window") & (m.split == "test")].iloc[40].to_dict()
model, device, ctype = infer.load_model(cfg["asr"])
rec = infer.transcribe(model, audio.load_unit_audio(row), cfg["asr"]["decode"])
print("REFERENCE :", row["reference"][:400])
print("WHISPER   :", rec["text"][:400])
print(review.assess(rec, domain="window"))